# Kinematics of the stellar tracers

The stars painted onto dark matter particles move with them, so the kinematics of the stellar halo are
weighted averages over the particles, using the weights $w_i$ of `stracers.compute_weights`:

$$\bar v = \frac{\sum_i w_i v_i}{\sum_i w_i}, \qquad
\sigma^2 = \frac{\sum_i w_i (v_i - \bar v)^2}{\sum_i w_i}.$$

This tutorial computes the velocities and velocity dispersions of two stellar haloes painted onto an
isotropic Hernquist halo, and checks them against the Jeans equation. With a simulation, use the
positions, velocities and potential of the dark matter particles instead of the toy halo.

In [ ]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

import stracers
from stracers.halos import sample_hernquist
from stracers.kinematics import spherical_velocities, bulk_velocity, velocity_profile
from stracers.plotting import LATEX_STYLE
from stracers.weights import G_GADGET as G

plt.rcParams.update(LATEX_STYLE)   # Computer Modern, text in math mode

## A halo and two stellar haloes

In [ ]:
n, M, a = 200_000, 1e10, 10.0            # particles, mass [Msun], scale radius [kpc]
pos, vel, pot = sample_hernquist(n, M, a, seed=1)
mass = M / n

tracers = {
    "Hernquist, a = 2 kpc": stracers.get_profile("hernquist", a=2.0),
    "Medina et al. 2024": stracers.mw.mw_profile("medina2024"),
}
# the cored (r_core = 1 kpc) Medina profile has a few negative f(E) bins at the
# most bound energies in this cuspy halo; compute_weights warns and zeroes them
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    weights = {name: stracers.compute_weights(pos, vel, pot, mass, nu, stellar_mass=1e8).weights
               for name, nu in tracers.items()}

## Velocities of the tracers

`spherical_velocities` gives the radial, polar and azimuthal components of each particle; the stars a
particle carries share its velocity. `bulk_velocity` is the weighted mean velocity of the stellar
halo, which should be ~0 here, since the halo is at rest at the origin.

In [ ]:
v_r, v_theta, v_phi = spherical_velocities(pos, vel)
w = weights["Medina et al. 2024"]
print("bulk velocity of the stellar halo [km/s]:", np.round(bulk_velocity(vel, w), 2))
print("mean radial velocity of the stars [km/s]: %.2f" % stracers.kinematics.weighted_mean(v_r, w))

## Velocity dispersion profiles

`velocity_profile` bins the particles in radius and returns, for each bin, the weighted mean velocities,
the dispersions $\sigma_r$, $\sigma_\theta$, $\sigma_\phi$, the anisotropy
$\beta = 1 - (\sigma_\theta^2 + \sigma_\phi^2)/2\sigma_r^2$, and the effective number of particles
$N_{\rm eff} = (\sum w)^2/\sum w^2$, which sets the statistical uncertainty,
$\delta\sigma \simeq \sigma/\sqrt{2 N_{\rm eff}}$. Without weights it gives the profile of the dark matter.

In [ ]:
edges = np.geomspace(0.5, 150.0, 13)
profiles = {"dark matter": velocity_profile(pos, vel, bins=edges)}
profiles.update({name: velocity_profile(pos, vel, w, bins=edges) for name, w in weights.items()})

p = profiles["Medina et al. 2024"]
print(" r [kpc]  sigma_r  sigma_theta  sigma_phi   beta   N_eff")
for row in zip(p.r, p.sigma_r, p.sigma_theta, p.sigma_phi, p.beta, p.n_eff):
    print("%8.1f %8.1f %12.1f %10.1f %6.2f %7.0f" % row)

## Comparison with the Jeans equation

For an isotropic tracer of density $\nu$ in a spherical potential, the Jeans equation gives
$$\sigma_r^2(r) = \frac{1}{\nu(r)} \int_r^\infty \nu(r')\, \frac{d\Phi}{dr'}\, dr',
\qquad \frac{d\Phi}{dr} = \frac{G M}{(r + a)^2}.$$
The more concentrated a stellar halo, the colder it is. The weighted dispersions follow the prediction
for each profile: the weights give the stars the right dynamics, not only the right density.

In [ ]:
def sigma_jeans(nu, r):
    return np.sqrt([quad(lambda x: nu(x) * G * M / (x + a)**2, ri, np.inf, limit=200)[0] / nu(ri) for ri in r])

nus = {"dark matter": stracers.get_profile("hernquist", a=a), **tracers}
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
rr = np.geomspace(0.5, 150, 200)
for (name, prof), color in zip(profiles.items(), ["0.4", "C0", "C3"]):
    err = prof.sigma_r / np.sqrt(2 * prof.n_eff)
    label = r"$\mathrm{" + name.replace(" ", r"\ ") + "}$"
    ax1.errorbar(prof.r, prof.sigma_r, err, fmt="o", ms=4, color=color, label=label)
    ax1.plot(rr, sigma_jeans(nus[name], rr), color=color, lw=1)
    ax2.errorbar(prof.r, prof.beta, fmt="o", ms=4, color=color)
ax1.plot([], [], "k-", lw=1, label=r"$\mathrm{Jeans\ equation}$")
ax1.set_xscale("log")
ax1.set_ylabel(r"$\sigma_r\ [\mathrm{km\,s^{-1}}]$")
ax1.legend(frameon=False)
ax2.axhline(0, color="k", lw=0.8, ls=":")
ax2.set_ylim(-0.5, 0.5)
ax2.set_xlabel(r"$r\ [\mathrm{kpc}]$")
ax2.set_ylabel(r"$\beta$");

## Rotation

A rotating stellar halo shows up as a non-zero mean azimuthal velocity $\bar v_\phi$. Here we add a
rotation of 30 km/s about the $z$ axis to all the particles and recover it:

In [ ]:
R = np.hypot(pos[:, 0], pos[:, 1])
vel_rot = vel + 30.0 * np.column_stack([-pos[:, 1] / R, pos[:, 0] / R, np.zeros(n)])
prof = velocity_profile(pos, vel_rot, weights["Medina et al. 2024"], bins=edges)
print("mean v_phi [km/s]:", np.round(prof.mean_vphi, 1))

## Tracer populations in a radial range

`stracers.paint_tracer` paints a population of a given tracer type, e.g. RR Lyrae or BHB stars, in a
radial range. It uses the most recent literature profile for that tracer whose fitted range covers the
range, computes the weights of the whole stellar halo, and keeps the particles in the range with their
number of stars. Here are 3000 RR Lyrae and 3000 BHB stars between 20 and 100 kpc:

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    pops = {t: stracers.paint_tracer(pos, vel, pot, mass, t, r_range=(20, 100), n_stars=3000)
            for t in ["RRLyr", "BHB"]}
for t, pop in pops.items():
    print(f"{t}: {pop.reference} ({pop.key}), fitted over {pop.fit_range[0]:g}-{pop.fit_range[1]:g} kpc, "
          f"{pop.total:.0f} stars")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for (t, pop), color in zip(pops.items(), ["C3", "C0"]):
    prof = pop.velocity_profile(bins=6)
    err = prof.sigma_r / np.sqrt(2 * prof.n_eff)
    label = r"$\mathrm{" + pop.tracer.replace(" ", r"\ ") + r"\ (" + pop.reference.replace(" ", r"\ ") + ")}$"
    ax.errorbar(prof.r, prof.sigma_r, err, fmt="o", ms=4, color=color, label=label)
    rr = np.geomspace(20, 100, 100)
    ax.plot(rr, sigma_jeans(pop.profile, rr), color=color, lw=1)
ax.set_xscale("log")
ax.set_xlabel(r"$r\ [\mathrm{kpc}]$")
ax.set_ylabel(r"$\sigma_r\ [\mathrm{km\,s^{-1}}]$")
ax.legend(frameon=False);

`pop.draw_stars(rng)` draws a mock catalogue (a Poisson number of stars per particle), returning the
particle index of each star:

In [ ]:
idx = pops["RRLyr"].draw_stars(rng=0)
print(len(idx), "mock RR Lyrae; first positions [kpc]:")
print(np.round(pops["RRLyr"].pos[idx[:3]], 1))

## In a simulation

- Pass the halo centre and its velocity as `center` and `vcenter` (e.g. from a shrinking-sphere centre),
  or recentre the particles first.
- To follow the stars to a later snapshot, map the weights onto its particles with
  `stracers.match_weights` and call `velocity_profile` with the positions and velocities of that snapshot.
- `weights * mass` (the stellar mass of each particle) can be used instead of the weights: the
  normalisation does not matter.